# 09 Gamma downstream forecasting study

Measures how the wrong RPF sign, and its correction by the held-out M9, changes direct seven-day-ahead net-load point forecasts on Gamma (Beta station B, one year). Three data conditions are compared: raw, M9-corrected and manually corrected.

Abbreviations used here: **RPF** is reverse power flow, the condition where a distribution substation exports power because rooftop solar exceeds local demand; a *wrong RPF sign* is a meter recording that stores the export as an import. **M7** is the deterministic threshold rule, **M8** the two-stage XGBoost classifier and **M9** the compact counterfactual method (revision 2). **MW** and **MWh** are megawatts and megawatt-hours; one interval is 15 minutes.

**Inputs.** `dataset/final/dataset_gamma.parquet` and the M9 decisions for beta_B from `06_site_days/site_days.parquet` (the fold that held beta_B out; its calibration never saw beta_B labels).

**Outputs.** `outputs/01_final_evaluation/09_gamma/`: the three series, data-error metrics, forecast predictions and metrics, the impact table, the correction audit and four figures; `manifests/09_gamma.json`.

**Approximate runtime.** About three minutes.

**Prerequisites.** Notebook 06.

**Main process.**

1. Apply the held-out M9 decisions to Gamma; days without a decision keep their raw values and are counted.
2. Build direct forecast examples (14-day history at the origin, calendar terms) with no observation after the origin.
3. Fit linear regression and XGBoost once per condition on targets before September 2024; seasonal naive needs no fit.
4. Score September 2024 against the manually corrected reference; draw the four figures.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display


def article_root() -> Path:
    """Locate publication/2_journal_article from JupyterLab, VS Code or the repository root."""
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "final_eval" / "cli.py").exists():
            return candidate
        nested = candidate / "publication" / "2_journal_article"
        if (nested / "final_eval" / "cli.py").exists():
            return nested
    raise FileNotFoundError("Could not locate publication/2_journal_article.")


ARTICLE = article_root()
sys.path.insert(0, str(ARTICLE))
sys.path.insert(0, str(ARTICLE.parents[1] / "src"))  # the repository's pynrpf package

from final_eval import cli, config  # noqa: E402

SETTINGS = config.load()  # verifies the frozen dataset hashes
OUT = SETTINGS.output_root()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("Article root:", ARTICLE.relative_to(ARTICLE.parents[2]))

## 2. Run the study

In [ ]:
result = cli.stage_gamma(SETTINGS)
display(result["audit"])
display(result["data_errors"].round(3))

## 3. Forecast impact

RMSE per model and data condition, and the reduction from raw to M9-corrected training data with the remaining gap to the manual reference.

In [ ]:
display(result["metrics"].round(3))
display(result["impact"].round(3))

## 4. Figures

In [ ]:
for name in ["fig01_gamma_raw_m9_manual_example_week", "fig02_gamma_data_error_rmse", "fig03_gamma_forecast_rmse",
             "fig04_gamma_forecast_residuals"]:
    display(Image(filename=OUT / "09_gamma" / f"{name}.png"))

## Conclusion

A single-substation case study: the effect is measured, not assumed, and the manually corrected condition remains label-dependent. The figures keep the format of the earlier Gamma study so the paper's section can be refreshed in place.